# Spark Bronze wikpedia page reads

In [ ]:

import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta import configure_spark_with_delta_pip

os.environ['SPARK_CONF_DIR'] = '/opt/tfds/spark/conf'

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg

import pyspark
def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    # spark_cfg = get_config("spark")

    #print(f"using s3 endpoint: {s3_cfg['url']}")
    #print(f"using spark master: {spark_cfg['master_url']}")

    conf = (
        pyspark.conf.SparkConf()
        # .setAppName("WhenDoIGetToSpark")

        # # the delta catalog works on top of the derby database we configure next.
        # .set(
        #     "spark.sql.catalog.spark_catalog",
        #     "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        # )
        # .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")

        # # there's no OSS catalog that works perfectly with delta. Hive spits out a lot of warnings about schema mergem but according to chatGPT and as we see, it all still works.
        # # these options are to make spark use a local derby database located in the mounted data directory. So, it will survive spark restarts and recreated dockers.
        # # I could not get the catalog to work while configured to use s3, local disk it is. data is still on s3.
        # .set("spark.sql.catalogImplementation", "hive")
        # .set("javax.jdo.option.ConnectionURL", "jdbc:derby:/opt/tfds/data/spark/metastore/metastore_db;create=true")
        # .set("javax.jdo.option.ConnectionDriverName", "org.apache.derby.jdbc.EmbeddedDriver")

        # # s3 configs
        .set("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .set("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        # .set("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])
        # # path.style.access seems necessary for using minio
        # .set("spark.hadoop.fs.s3a.path.style.access", "true")

        # # this makes the delta tables go to s3
        # .set("spark.sql.warehouse.dir", "s3a://dwh/warehouse/")

        # # default is 200 partitions, which apparently is a bit high for a local setup
        # .set("spark.sql.shuffle.partitions", "12")

        # # the wikipedia pageview dataset would not go through on default level. we need a bit more memory.
        # .set("spark.executor.memory", "2g")

        # # if we don't tell pyspark we have a cluster it will do it all locally in notebook
        # .setMaster(spark_cfg['master_url'])
        # # it works fine, and is great for troubleshooting.
        # # You first make your config run locally, then you try to get the cluster to run the same thing.
        # .setMaster("local[*]")
    )

    # these are for using s3
    # extra_packages = [
    #     "org.apache.hadoop:hadoop-aws:3.3.4",
    #     "org.apache.hadoop:hadoop-common:3.3.4",
    #     "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    # ]

    builder = pyspark.sql.SparkSession.builder.config(conf=conf)
    spark_session = configure_spark_with_delta_pip(
        builder #, extra_packages=extra_packages
    ).getOrCreate()


    return spark_session

def show_cfg(spark_session):
    cfg = spark_session.sparkContext.getConf().getAll()
    for key, value in cfg:
        if key in (
            'spark.submit.pyFiles',
            'spark.driver.extraJavaOptions',
            'park.app.initial.jar.urls',
            'spark.files',
            'spark.repl.local.jars',
            'spark.app.initial.file.urls'
            'spark.executor.extraJavaOption',
            'spark.app.initial.jar.urls'
            'spark.app.initial.file.urls'
            ):
            print(key)
            for l in value.split(','):
                print('    ' + str(l))
        else:
            print(f'{key} = {value}')

def show_spark(spark):
    cfg = spark.sparkContext.getConf()
    print(f'using spark master: {cfg.get("spark.master")}')
    print(f'Delta lake location: {cfg.get("spark.sql.warehouse.dir")}')
    print(f'S3 endpoint: {cfg.get("spark.hadoop.fs.s3a.endpoint")}')

In [ ]:
from pyspark.sql.functions import input_file_name, col, sum as _sum, substring, to_date

def load_page_reads(s3_path):
    schema = StructType([
        StructField(name="domain_code", dataType=StringType(), nullable = True),
        StructField("page_title", StringType(), True),
        StructField("count_views", StringType(), True)
    ])
    spark.sparkContext.setLogLevel("WARN")


    print(f'Loading data:{s3_path}')
    data = (
        spark.read.format("csv")
        .option("delimiter", " ")
        .option("header", "false")
        .option("inferSchema", "false")
        .schema(schema)
        .load(s3_path)
    )
    print('Files loaded:')
    for f in data.inputFiles():
        print(f)
    return data

def enrich_page_reads(data):
    data_enriched = (
        data
        .na.drop(subset=["domain_code"])
        .filter(~col("page_title").contains(":"))
        .filter(~col("page_title").isin("-", 'Main_Page', 'Forside', 'Hauptseite', 'wiki.phtml'))
        .withColumn("country_code", substring(col("domain_code"), 1, 2))
        .filter(col("domain_code").isin("sv", 'dk', 'no', 'de', 'en'))
        .withColumn("count_views", col("count_views").cast(IntegerType()))
        .withColumn("file_name", input_file_name())
        .withColumn("date", to_date(substring(col("file_name"), -18, 8), 'yyyyMMdd'))
    )
    return data_enriched

def write_page_reads(data):

    spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
    (   data
        .write
        .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
        .option("mergeSchema", "true")
        .format("delta")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
        .partitionBy("date")
        .saveAsTable("bronze.wikipedia_page_reads")
    )
    print(spark.catalog.listDatabases())
    print(spark.catalog.listTables("bronze"))



spark = get_spark_session()
show_spark(spark)
spark.sparkContext.setLogLevel("WARN")

# spark.catalog.clearCache()


s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-220000.gz"
s3_path = "s3a://data/test.csv"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/*/*.gz"

data_raw = load_page_reads(s3_path=s3_path)
data_enriched = enrich_page_reads(data_raw)
write_page_reads(data=data_enriched)
spark.stop()
print('all done')


## Bronze
Performs: 
* ingestion
* column naming
* column casting
* get the date from the filename
* partitioning

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
spark = get_spark_session()
aggregated_df = (
    spark.table("bronze.wikipedia_page_reads")
    .groupBy('date', "page_title", "country_code")
    .agg(
        _sum(col("count_views")).alias("total_count_views"),
    )
)

national_win = (
    Window
    .partitionBy('date', "country_code")
    .orderBy(col("total_count_views").desc())
)

ranked_df = (
    aggregated_df
    .withColumn("national_rank", row_number().over(national_win))
    )

final_df = (
    ranked_df
    .filter(col('national_rank') <= 3)
    .orderBy('date', "national_rank", "country_code")
)

# final_df.explain(mode="extended")

In [ ]:
final_df.show(50, truncate=False)

In [3]:
import json
import os
def list_tables(prompt):
    print('-'*50)
    print(prompt)
    dbs = spark.catalog.listDatabases()

    print("databases:")
    print(json.dumps(dbs, indent=4))
    for db in dbs:
        if db.name == "bronze":
            print ("tables")
            tbls =spark.catalog.listTables("bronze")
            print(json.dumps(tbls, indent=4))
            break
    else:
        print('no tables; bronze database missing')


# ranked_df.explain(mode="extended")
# spark.stop()

# make sure we have a fresh spark sessions
os.environ['SPARK_CONF_DIR'] = '/opt/tfds/spark/conf'

spark = get_spark_session()

spark.stop()

spark = get_spark_session()
show_cfg(spark)
list_tables("after first re-start")



# spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
# list_tables("after creating bronze")

# data = spark.range(10)
# spark.sql("DROP TABLE IF EXISTS bronze.test_table")
# (   data
#         .write
#         .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
#         .option("mergeSchema", "true")
#         .format("delta")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
#         .saveAsTable("bronze.test_table")
# )
# list_tables("after creating bronze.test_table")

# spark.stop()
# spark = get_spark_session()
# list_tables("after second re-start")


retrieving s3 config from http://tfds-config:8005/api/configs/s3


25/04/23 19:37:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


retrieving s3 config from http://tfds-config:8005/api/configs/s3
spark.app.startTime = 1745429867667
spark.hadoop.fs.s3a.path.style.access = true
spark.jars.packages = io.delta:delta-spark_2.12:3.3.0
spark.hadoop.fs.s3a.access.key = MAVE5IoQGj5zYKLwANfp
spark.files
    file:///Users/jens/.ivy2/jars/io.delta_delta-spark_2.12-3.3.0.jar
    file:///Users/jens/.ivy2/jars/io.delta_delta-storage-3.3.0.jar
    file:///Users/jens/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar
spark.repl.local.jars
    file:///Users/jens/.ivy2/jars/io.delta_delta-spark_2.12-3.3.0.jar
    file:///Users/jens/.ivy2/jars/io.delta_delta-storage-3.3.0.jar
    file:///Users/jens/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar
spark.sql.shuffle.partitions = 12
spark.driver.host = 192.168.0.152
spark.hadoop.fs.s3a.secret.key = QqJ956F45jD5pTd3beIRrm02bnmksXsbkTgSfXvY
spark.master = spark://spark-master:7077
spark.serializer.objectStreamReset = 100
spark.submit.pyFiles
    /Users/jens/.ivy2/jars/io.delta_delta-spark_2.12-3.

databases:
[
    [
        "bronze",
        "spark_catalog",
        "",
        "s3a://dwh/warehouse/bronze.db"
    ],
    [
        "default",
        "spark_catalog",
        "Default Hive database",
        "s3a://dwh/warehouse"
    ]
]
tables


25/04/23 19:37:50 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


[
    [
        "test_table",
        "spark_catalog",
        [
            "bronze"
        ],
        null,
        "MANAGED",
        false
    ],
    [
        "wikipedia_page_reads",
        "spark_catalog",
        [
            "bronze"
        ],
        null,
        "MANAGED",
        false
    ]
]


In [ ]:
import os
print(os.environ.get("SPARK_CONF_DIR"))